<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Prompt Engineering für Agenten
</b></font> </br></p>

---

In [ ]:
#@title  🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M04-Prompt-Engineering"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---



In diesem Modul lernen Sie, wie Prompts das Verhalten von Agenten steuern.


In [ ]:
# Imports für das Modul
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

# 2 | ChatPromptTemplate
---

`ChatPromptTemplate` trennt klar zwischen Rollen (`system`, `human`, `ai`) und Variablen.
Das ist robuster als Prompt-Strings zusammenzukleben.


**`load_prompt()` – Prompts als versionierte Assets**

Statt Prompts direkt im Code zu schreiben, laedt `load_prompt()` ein Markdown-Dokument aus einem lokalen Pfad oder einer GitHub-URL und gibt ein fertiges `ChatPromptTemplate` zurueck.

```python
# Aus GitHub (Blob- und Tree-URLs werden automatisch in Raw-Links umgewandelt)
prompt = load_prompt(
    "https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_python_tutor_prompt.md",
    mode="T"   # "T" = ChatPromptTemplate | "S" = plain String
)

# Oder lokal:
prompt = load_prompt("../05_prompt/m04_python_tutor_prompt.md", mode="T")
```



**Prompt-Format (Markdown mit YAML-Frontmatter):**
```markdown
---
name: python_tutor
description: Erklaert Python-Konzepte fuer verschiedene Zielgruppen
variables: [thema, zielgruppe]
---

## system
Du bist ein Python-Tutor fuer {zielgruppe}.

## human
Erklaere {thema} mit einem Beispiel.
```



**Vorteile gegenüber Inline-Prompts:**

| | Inline im Code | `load_prompt()` aus Datei |
|---|---|---|
| **Versionierung** | Im Code-Commit versteckt | Eigener Commit, diff-bar |
| **Uebersichtlichkeit** | Code und Prompts vermischt | Klar getrennt |
| **Wiederverwendung** | Copy-Paste | Eine Datei, viele Notebooks |
| **Zusammenarbeit** | Erfordert Code-Kenntnisse | Einfaches Markdown |

> Das `05_prompt/`-Verzeichnis im Repository ist damit ein **Prompt Hub** – alle Prompts an einem Ort, versioniert per Git. Alle Notebooks dieses Kurses nutzen dieses Muster.

In [ ]:
prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_python_tutor_prompt.md", mode="T")

messages = prompt.invoke({
    "thema": "for-Schleifen",
    "zielgruppe": "Einsteiger"
})

for msg in messages.messages:
    print(f"{msg.type.upper()}: {msg.content}")

In [ ]:
chain = prompt | llm
response = chain.invoke({"thema": "List Comprehension", "zielgruppe": "Data-Analysten"})
mprint(response.content)

# 3 | System Prompts für Tool-Calling
---



Für Agenten mit Tools sollte der `system`-Prompt explizit festlegen:
- wann ein Tool verwendet wird
- wann **kein** Tool verwendet wird
- welches Ausgabeformat erwartet wird


In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def add(a: float, b: float) -> float:
    """Addiert zwei Zahlen und gibt das Ergebnis zurueck."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """Multipliziert zwei Zahlen und gibt das Ergebnis zurueck."""
    return a * b

system_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_rechenassistent_system_prompt.md", mode="S")

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[add, multiply],
    system_prompt=system_prompt,
)

result = agent.invoke({"messages": [{"role": "user", "content": "(12 + 8) * 3"}]})
mprint(result["messages"][-1].content)


# 4 | Few-Shot Examples
---



Few-Shot-Beispiele helfen, Stil und Struktur der Antworten zu stabilisieren.
Nutzen Sie wenige, aber repräsentative Beispiele.


In [ ]:
zero_shot = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_ticket_zero_shot_prompt.md", mode="T")
few_shot = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_ticket_few_shot_prompt.md", mode="T")
for name, p in [("Zero-Shot", zero_shot), ("Few-Shot", few_shot)]:
    # 1. Tracing-Konfiguration vorab festlegen
    run_cfg = {
        "run_name": f"M04_Kap4_{name.replace('-', '')}",
        "tags": ["M04", "few-shot", name.lower().replace("-", "")]
    }
    # 2. with_config() anwenden, dann invoke()
    out = (p | llm).with_config(**run_cfg).invoke({"ticket": "Wo finde ich meine letzte Rechnung?"})
    print()
    print(f"{name}: {out.content.strip()}")


# 5 | Prompt-Varianten testen
---



Ein einfacher Vergleich hilft, bessere Prompts datenbasiert auszuwählen.
Unten testen wir mehrere System-Prompts auf dieselbe Anfrage.


In [ ]:
variants = {
    "knapp": "Antworte in maximal 2 Saetzen.",
    "didaktisch": "Erklaere schrittweise mit kurzem Beispiel.",
    "kritisch": "Zeige Antwort und nenne eine moegliche Fehlerquelle.",
}
from langchain_core.prompts import ChatPromptTemplate
variant_prompt = ChatPromptTemplate.from_messages([
    ("system", "Du bist Python-Coach. {instruction}"),
    ("human", "{q}"),
])
user_question = "Wann sollte ich in Python eine Liste statt eines Sets verwenden?"
for name, instruction in variants.items():
    # 1. Tracing-Konfiguration vorab festlegen
    run_cfg = {
        "run_name": f"M04_Kap5_Variant_{name}",
        "tags": ["M04", "prompt-variants", name]
    }
    # 2. with_config() anwenden, dann invoke()
    answer = (variant_prompt | llm).with_config(**run_cfg).invoke({"instruction": instruction, "q": user_question})
    mprint("### Variante: " + name)
    mprint('---')
    mprint(answer.content.strip())


# 6 | Strukturierte Agenten-Prompts: Production Pattern
---

LangGraph-Produktions-Prompts folgen einem bewährten Muster mit 5 Patterns,
die Agenten zuverlässiger und kontrollierbarer machen.

## Pattern 1: XML-Tags als Struktur-Trenner

LLMs folgen klar abgegrenzten Sektionen zuverlässiger als Fließtext.
Jede Sektion hat einen eigenen kognitiven Zweck.

```python
# ❌ Unstrukturiert – LLM muss Regeln aus Text extrahieren
system = "Du bist Assistent. Nutze Tools. Sei effizient. Stoppe wenn fertig."

# ✅ Strukturiert – klare Trennung von Rolle, Prozess und Grenzen
system = """
<Task>Was der Agent tun soll</Task>
<Instructions>Wie er vorgehen soll</Instructions>
<Hard Limits>Wann er stoppen soll</Hard Limits>
"""
```

## Pattern 2: Explizite Tool-Steuerung

Ohne Anweisung nutzen Agenten Tools opportunistisch – springen direkt zur nächsten Aktion.
Explizite Reflexions-Aufforderung erzwingt Bewertung nach jedem Schritt.

```python
"KRITISCH: Bewerte nach jedem Tool-Ergebnis: Habe ich genug um zu antworten?"
```

## Pattern 3: Quantitative Limits statt qualitativer Anweisungen

```python
# ❌ Qualitativ – nicht handlungsleitend für ein LLM
"Sei sparsam mit Tool-Calls."

# ✅ Quantitativ – konkret und messbar
"Tool-Budget: maximal 3 Suchen. Nach 3 Suchen immer stoppen."
```

## Pattern 4: Explizite Stop-Bedingungen

Agenten neigen zu Over-Research. Ohne Stop-Kriterien laufen sie bis zum recursion_limit.

```python
"""Sofort stoppen wenn:
- Du die Frage vollständig beantworten kannst
- Die letzte Suche keine neuen Informationen brachte
- Du 3+ relevante Quellen gefunden hast
"""
```

## Pattern 5: Strategie vorgeben (Breit → Eng)

Ein mentales Modell der Vorgehensweise ist wirkungsvoller als ein Regelwerk.

```python
"""<Instructions>
1. Lies die Anfrage sorgfältig
2. Starte mit breiteren Suchanfragen (allgemeines Thema zuerst)
3. Verfeinere mit spezifischeren Suchen wenn Infos fehlen
4. Stoppe sobald du die Anfrage beantworten kannst
</Instructions>
"""
```

**Beispiel unten:** Alle 5 Patterns in einem praktischen Agenten-Prompt.

In [ ]:
# Strukturierter System-Prompt mit allen 5 Patterns
from langchain_core.tools import tool

structured_system_prompt = """Du bist ein Python-Tutor fuer den KI-Agenten-Kurs.

<Task>
Beantworte Python-Fragen klar, praxisnah und mit Beispielen.
</Task>

<Instructions>
1. Lies die Frage sorgfaeltig – Grundlagen oder fortgeschrittenes Thema?
2. Beginne mit dem Kernkonzept, dann ein konkretes Beispiel
3. Pruefe: Ist die Erklaerung fuer die Zielgruppe verstaendlich?
4. Ergaenze einen Praxistipp wenn hilfreich
</Instructions>

<Hard Limits>
Maximale Antwortlaenge: 150 Woerter.
Sofort antworten wenn: Kernkonzept und Beispiel erklaert sind.
</Hard Limits>"""

@tool
def python_beispiel(konzept: str) -> str:
    """Gibt ein konkretes Python-Code-Beispiel fuer ein Konzept zurueck."""
    beispiele = {
        "list comprehension": "[x**2 for x in range(10) if x % 2 == 0]",
        "lambda":             "quadrat = lambda x: x ** 2",
        "decorator":          "@functools.wraps(func)",
        "generator":          "def zahlen(): yield from range(10)",
    }
    return beispiele.get(konzept.lower(), f"# Beispiel fuer: {konzept}\npass")

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[python_beispiel],
    system_prompt=structured_system_prompt,
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Erkläre List Comprehensions fuer Einsteiger."}]
})
mprint(result["messages"][-1].content)

# 7 | LangSmith: Prompt-Inspektion
---



LangSmith macht Prompt-Experimente nachvollziehbar: Welcher Prompt erzeugte welche Antwort?

**Empfehlung im Kursbetrieb:**
- `run_name` pro Experiment setzen
- `tags` für Modul/Kapitel verwenden
- bei Vergleichen immer denselben Input nutzen

**Hinweis (EU):** Endpoint `https://eu.api.smith.langchain.com` – bereits in der Setup-Cell konfiguriert.

In [ ]:
# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M04_Kap6_PromptBaseline",
    "tags":     ["M04", "prompt-engineering"]
}

# 2. with_config() anwenden, dann invoke()
traced_chain = (prompt | llm).with_config(**run_cfg)
traced_response = traced_chain.invoke({
    "thema": "while-Schleifen",
    "zielgruppe": "Python-Einsteiger"
})

mprint("### Trace erstellt")
mprint('---')
mprint(traced_response.content)

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M04-Prompt-Engineering", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.



1. Erstellen Sie einen Agenten mit 2-3 Tools (z. B. Rechnen, Datum, String-Utils).
2. Schreiben Sie zwei unterschiedliche System-Prompts: einmal "knapp", einmal "didaktisch".
3. Definieren Sie 3 Testfragen und vergleichen Sie beide Varianten.
4. Dokumentieren Sie kurz: Welche Prompt-Version ist für Ihren Use Case besser und warum?

